# Cuando el camino más corto no es el mejor
**Fundamentos de Inteligencia Artificial — Grupo LITIID71**

Prototipo de búsqueda de rutas en una zona inundada: un agente debe llegar desde una zona afectada hasta un refugio, evitando zonas bloqueadas y considerando el riesgo de cada zona según su nivel de inundación.

| Integrante | Rol |
|---|---|
| Hector Partido Flores | Coordinación e integración |
| Ismael Alexander Hidalgo Aguilando | Terreno y visualización |
| Hector Martínez Sánchez | Búsqueda no informada |
| Gildardo Emmanuel González García | Búsqueda informada y comparación |
| Anthony Oziel Ramos López | Validación y pruebas |

## 0. Configuración
Importaciones y contrato común del proyecto (`src/modelo.py`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.modelo import *

print("Entorno listo")

## 1. Introducción y problema
_Responsable: Hector Partido Flores_

- Descripción del escenario de inundación.
- Diferencia entre **encontrar una ruta** y **encontrar una ruta conveniente**.
- Pregunta guía: si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?

## 2. Modelo del terreno y función de costo
_Responsable: Ismael Alexander Hidalgo Aguilando_

- Representación del terreno como cuadrícula de niveles de inundación (0 libre, 1 leve, 2 media, 3 bloqueada).
- Movimiento en 4 direcciones.
- Costo de entrar a una celda: `1 + λ · riesgo(nivel)`.
- Justificación de los valores de riesgo elegidos.
- Escenarios de prueba.

## 2. Modelo del terreno y función de costo
_Responsable: Ismael Alexander Hidalgo Aguilando_

### 2.1 Representación del terreno
El terreno se representa como una **cuadrícula** (matriz de NumPy) donde cada celda es una zona y guarda su nivel de inundación.

Se eligió la cuadrícula porque:
- Permite construir y modificar escenarios fácilmente: bloquear una zona o cambiar su riesgo es cambiar un rectángulo de la matriz (`colocar_zona`).
- Se puede dibujar directamente como imagen, con un color por nivel.
- Cada celda tiene como máximo 4 vecinos, así que el problema es fácil de seguir a mano al explicar los algoritmos.

Un grafo general (calles e intersecciones) sería más realista, pero para comparar estrategias de búsqueda la cuadrícula basta y hace los resultados más fáciles de verificar.

### 2.2 Movimientos
El agente se mueve en **4 direcciones** (arriba, abajo, izquierda, derecha), sin diagonales:
- Todos los pasos miden lo mismo (1), así que contar pasos es medir distancia.
- Se evita "cortar esquinas" pasando en diagonal entre dos celdas bloqueadas.
- La distancia Manhattan nunca sobreestima los pasos necesarios, lo que la hace una heurística admisible para A\*.

### 2.3 Niveles de inundación y riesgo

| Nivel | Significado | Transitable | Riesgo |
|---|---|---|---|
| 0 | Seco | Sí | 0 |
| 1 | Inundación leve | Sí | 1 |
| 2 | Inundación media | Sí | 3 |
| 3 | Inundación severa | **No** | — |

- **El riesgo no crece de forma lineal (0, 1, 3).** Pasar de agua leve a agua media es mucho más peligroso que pasar de seco a leve: hay más profundidad y corriente. Por eso una celda media pesa el triple que una leve.
- **La inundación severa es una restricción, no un costo alto.** Se trata como zona bloqueada: ningún algoritmo puede pisarla, sin importar cuánto ahorre. Así se distinguen dos tipos de condición:
  - **restricciones:** lo que nunca se permite;
  - **penalizaciones:** lo que se permite, pero cuesta más.

### 2.4 Función de costo
El costo de **entrar** a una celda es:

$$\text{costo de paso} = 1 + \lambda \cdot \text{riesgo}(\text{celda})$$

y el costo total de una ruta es:

$$\text{costo de la ruta} = \text{pasos} + \lambda \cdot \text{riesgo acumulado}$$

- El **1** representa la distancia. Garantiza que todo paso cueste al menos 1, así que ninguna ruta es "gratis".
- **λ** indica cuánto pesa la seguridad frente a la distancia:
  - con **λ = 0** solo importan los pasos y se obtiene la ruta más corta;
  - al **aumentar λ**, cruzar agua se vuelve caro y el algoritmo prefiere rodearla.

**Respuesta a la pregunta guía:** si dos rutas llegan al mismo refugio, el sistema no elige la más corta ni la más segura por separado. Calcula el costo total de cada una con esta fórmula y elige la de **menor costo**. La ruta corta gana solo si el riesgo que ahorra la ruta segura no compensa los pasos extra.

In [ ]:
from src.terreno import crear_terreno, colocar_zona
from src.escenarios import escenario_atajo_inundado, escenario_dos_refugios
from src.visualizacion import dibujar_terreno, comparar_rutas, leyenda, NOMBRES_NIVEL

# Tabla del modelo: riesgo y costo de entrar a cada tipo de celda con λ = 1
pd.DataFrame([
    {
        "Nivel": NOMBRES_NIVEL[n],
        "Transitable": "Sí" if n != BLOQUEADA else "No",
        "Riesgo": RIESGO_POR_NIVEL.get(n, "—"),
        "Costo de paso (λ=1)": 1 + RIESGO_POR_NIVEL[n] if n in RIESGO_POR_NIVEL else "—",
    }
    for n in (LIBRE, LEVE, MEDIA, BLOQUEADA)
])

### 2.5 ¿Cuántos pasos extra vale la pena caminar para evitar el agua?
Como cada paso cuesta 1, el término `λ · riesgo` se puede leer como **pasos extra**. Evitar una celda inundada vale la pena si el rodeo agrega menos pasos que `λ · riesgo` de esa celda.

In [ ]:
equivalencias = []
for lam in (0, 0.5, 1, 2, 5):
    fila = {"λ": lam}
    for n in (LEVE, MEDIA):
        fila[f"{NOMBRES_NIVEL[n]}: costo de paso"] = 1 + lam * RIESGO_POR_NIVEL[n]
        fila[f"{NOMBRES_NIVEL[n]}: pasos extra aceptados"] = lam * RIESGO_POR_NIVEL[n]
    equivalencias.append(fila)

pd.DataFrame(equivalencias).set_index("λ")

**Cómo leer la tabla:**
- Con **λ = 0**, el agua no cuesta nada extra: el algoritmo nunca da un rodeo para evitarla.
- Con **λ = 1**, evitar una celda de inundación media justifica caminar hasta **3 pasos extra**, y una leve hasta **1 paso extra**.
- Con **λ = 5**, el sistema es muy conservador: acepta hasta 15 pasos extra para no pisar una celda media.

Así, λ es un parámetro que se puede ajustar según la situación. Por ejemplo, con una evacuación nocturna o con personas vulnerables convendría un λ alto. En la sección 5.1 se muestra con ejecuciones reales cómo cambia la ruta elegida al variar λ.

In [ ]:
terreno, inicio, refugios = escenario_atajo_inundado()
fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
dibujar_terreno(terreno, inicio, refugios, titulo="Escenario principal: atajo inundado", ax=ax)
leyenda(fig, con_ruta=False)
plt.show()

## 3. Estrategias de búsqueda

Todas las estrategias reciben lo mismo (`terreno, inicio, refugios, lam`) y devuelven un `Resultado` con la misma estructura. Para que la comparación sea justa, todas miden el trabajo con los mismos criterios:

| Métrica | Qué mide |
|---|---|
| **Nodos generados** | Cuántas celdas se agregaron a la frontera (incluye el inicio). |
| **Nodos expandidos** | Cuántas celdas se sacaron de la frontera y se revisaron sus vecinos. Es la medida principal de trabajo. |
| **Frontera máxima** | El mayor número de celdas pendientes al mismo tiempo. Indica cuánta memoria usó la búsqueda. |
| **Tiempo** | Milisegundos medidos con `time.perf_counter()`. Varía entre ejecuciones y equipos. |

En todas, la prueba de meta se hace **al sacar** la celda de la frontera, no al agregarla. Así las métricas son comparables entre algoritmos.

### 3.1 Búsqueda no informada (BFS, DFS, UCS)
_Responsable: Hector Martínez Sánchez_

Las estrategias no informadas no saben dónde está el refugio: solo exploran a partir del inicio. La diferencia entre ellas es **qué celda de la frontera eligen expandir primero**.

| Algoritmo | Estructura de la frontera | Expande primero | ¿Usa el riesgo para decidir? | Garantiza |
|---|---|---|---|---|
| **BFS** (anchura) | Cola (FIFO) | La celda que lleva más tiempo esperando | No | La ruta con **menos pasos** |
| **DFS** (profundidad) | Pila (LIFO) | La celda agregada más recientemente | No | Solo **encontrar una ruta**, si existe |
| **UCS** (costo uniforme) | Cola de prioridad por costo acumulado `g` | La celda con menor costo acumulado | **Sí** | La ruta de **menor costo** |

- **BFS** avanza por "capas": primero todas las celdas a 1 paso, luego a 2, y así sucesivamente. Por eso la primera vez que llega al refugio lo hace con el mínimo de pasos, aunque cruce agua.
- **DFS** sigue un camino hasta el fondo antes de regresar. Encuentra una ruta, pero puede ser muy larga y dar muchas vueltas. Se incluye como contraste: **encontrar una ruta no es lo mismo que encontrar una ruta conveniente.**
- **UCS** siempre expande la celda más barata hasta el momento, con costo `1 + λ·riesgo` por paso. Es la única de las tres que considera el riesgo al decidir.

En BFS y DFS, λ no cambia la ruta: solo se usa para **reportar** cuánto cuesta la ruta que encontraron.

In [ ]:
from src.escenarios import escenario_atajo_inundado
from src.busqueda_no_informada import bfs, dfs, ucs
from src.validacion import validar_resultado


def tabla_resultados(resultados, terreno, inicio, refugios, lam):
    """Convierte una lista de Resultado en una tabla y valida cada ruta."""
    filas = []
    for r in resultados:
        valida, _ = validar_resultado(terreno, inicio, refugios, r, lam)
        filas.append({
            "Algoritmo": r.algoritmo,
            "Encontró ruta": "Sí" if r.encontrada else "No",
            "Ruta válida": "Sí" if valida else "No",
            "Costo": r.costo,
            "Pasos": r.pasos,
            "Riesgo acumulado": r.riesgo_acumulado,
            "Nodos expandidos": r.nodos_expandidos,
            "Nodos generados": r.nodos_generados,
            "Frontera máx.": r.frontera_max,
            "Tiempo (ms)": round(r.tiempo_ms, 3),
        })
    return pd.DataFrame(filas).set_index("Algoritmo")


terreno, inicio, refugios = escenario_atajo_inundado()
resultados_no_inf = [alg(terreno, inicio, refugios, lam=1.0) for alg in (bfs, dfs, ucs)]
tabla_resultados(resultados_no_inf, terreno, inicio, refugios, lam=1.0)

#### ¿Qué pasa si el riesgo no importa (λ = 0)?
Con λ = 0 todos los pasos cuestan 1, así que el costo de una ruta es igual a su número de pasos. En ese caso UCS debería comportarse como BFS.

In [ ]:
bfs_0 = bfs(terreno, inicio, refugios, lam=0)
ucs_0 = ucs(terreno, inicio, refugios, lam=0)

display(tabla_resultados([bfs_0, ucs_0], terreno, inicio, refugios, lam=0))
print("¿Misma ruta exacta?", bfs_0.ruta == ucs_0.ruta)

**Observaciones**

- **BFS** encuentra la ruta con menos pasos (11), pero cruza el centro de la inundación: riesgo acumulado 14, costo 25.
- **UCS** da 8 pasos más (19), pero con riesgo 0 y costo 19. Es la ruta más conveniente con λ = 1.
- **DFS** encuentra una ruta válida, pero es la peor: 83 pasos, costo 151 y la frontera más grande (72). Demuestra que encontrar una ruta no basta.
- UCS hizo un poco más de trabajo que BFS (89 contra 81 nodos expandidos) porque tuvo que explorar los rodeos. Considerar el riesgo tiene un costo computacional, aunque aquí es pequeño.
- Con **λ = 0**, UCS y BFS dan exactamente la misma ruta y el mismo trabajo. Esto confirma que la diferencia entre ambos viene **solo** del riesgo: UCS es una generalización de BFS en la que los pasos pueden tener distinto costo.

### 3.2 Búsqueda informada (Greedy Best-First, A*)
_Responsable: Gildardo Emmanuel González García_

Las estrategias informadas usan una **heurística** `h(n)`, que es una estimación de cuánto falta para llegar al refugio. Con ella deciden qué celda expandir primero.

**Heurística utilizada: distancia Manhattan al refugio más cercano**

$$h(n) = \min_{r \in \text{refugios}} \big(|fila_n - fila_r| + |columna_n - columna_r|\big)$$

Esta heurística es **admisible**, es decir, nunca sobreestima el costo real, porque:
1. Sin movimientos diagonales se necesitan **al menos** `|Δfila| + |Δcolumna|` pasos para llegar.
2. Cada paso cuesta **como mínimo 1** (cuando el riesgo es 0), así que el costo real es mayor o igual que el número de pasos.

También es **consistente**: entre dos celdas vecinas, `h` cambia como máximo en 1, y cada paso cuesta al menos 1. Esto garantiza que A\* con conjunto de cerrados encuentre la ruta óptima.

| Algoritmo | Ordena la frontera por | ¿Usa el riesgo para decidir? | Garantiza |
|---|---|---|---|
| **Greedy Best-First** | `h(n)`: lo que *parece* más cercano | No | Nada; puede dar una ruta mala |
| **A\*** | `f(n) = g(n) + h(n)`: costo real acumulado + estimación | **Sí**, a través de `g` | La ruta de **menor costo** (igual que UCS) |

- **Greedy** solo mira hacia el refugio. Avanza en línea recta aunque haya agua, porque ignora lo que lleva gastado.
- **A\*** combina el costo real con riesgo (`g`, como UCS) con la heurística (`h`, como Greedy). Obtiene la misma ruta óptima que UCS, pero la heurística le permite descartar direcciones que se alejan del refugio y **expandir menos nodos**.

In [ ]:
from src.busqueda_informada import greedy, a_estrella

resultados_inf = [alg(terreno, inicio, refugios, lam=1.0) for alg in (greedy, a_estrella)]
tabla_resultados(resultados_inf, terreno, inicio, refugios, lam=1.0)

**Observaciones**

- **Greedy** expande solo 11 nodos, el mínimo posible para una ruta de 11 pasos: va directo al refugio. Pero cruza el centro de la inundación y su costo es 25, igual que BFS.
- **A\*** obtiene la misma ruta óptima que UCS (costo 19, riesgo 0), pero expande **48 nodos en lugar de 89**: casi la mitad del trabajo. Esa es la ventaja de una heurística admisible.
- Usar una heurística no basta para obtener rutas convenientes. Lo que importa es **cómo se combina** con el costo real: Greedy la usa sola, A\* la suma al costo con riesgo.

## 4. Visualización del terreno y las rutas
_Responsable: Ismael Alexander Hidalgo Aguilando_

- Mapa con colores por nivel de inundación.
- Rutas de cada algoritmo dibujadas sobre el terreno.

In [ ]:
from src.busqueda_no_informada import bfs, dfs, ucs
from src.busqueda_informada import greedy, a_estrella

ALGORITMOS = [bfs, dfs, ucs, greedy, a_estrella]

resultados = [alg(terreno, inicio, refugios, lam=1.0) for alg in ALGORITMOS]
fig = comparar_rutas(terreno, inicio, refugios, resultados)
fig.suptitle("Escenario principal (λ = 1)", x=0.01, ha="left", fontsize=11)
plt.show()

In [ ]:
terreno_2, inicio_2, refugios_2 = escenario_dos_refugios()
resultados_2 = [alg(terreno_2, inicio_2, refugios_2, lam=1.0) for alg in ALGORITMOS]
fig = comparar_rutas(terreno_2, inicio_2, refugios_2, resultados_2)
fig.suptitle("Dos refugios: uno cercano tras el agua y otro lejano en seco (λ = 1)",
             x=0.01, ha="left", fontsize=11)
plt.show()

**Observaciones**

- En el escenario principal, BFS y Greedy cruzan la zona inundada: 11 pasos, pero costo 25. UCS y A* la rodean por el norte: 19 pasos, costo 19. La ruta más corta no es la más conveniente.
- DFS encuentra una ruta, pero es muy larga y atraviesa agua varias veces. Encontrar una ruta no es lo mismo que encontrar una ruta conveniente.
- En el escenario de dos refugios, BFS y Greedy eligen el refugio cercano cruzando el agua (costo 16). UCS y A* eligen el refugio lejano por terreno seco (costo 15).

## 5. Comparación de algoritmos
_Responsable: Gildardo Emmanuel González García_

Se ejecutan los cinco algoritmos en los dos escenarios con λ = 1.

- Todas las cifras de costo y trabajo salen de ejecuciones reales en este notebook.
- Como el tiempo varía entre ejecuciones, se reporta la **mediana de 30 ejecuciones** de cada algoritmo.
- Cada ruta se comprueba con el validador independiente (columna "Ruta válida").

In [ ]:
import statistics
from src.escenarios import escenario_atajo_inundado, escenario_dos_refugios

ALGORITMOS = [bfs, dfs, ucs, greedy, a_estrella]
ESCENARIOS = {
    "Atajo inundado": escenario_atajo_inundado,
    "Dos refugios": escenario_dos_refugios,
}
REPETICIONES = 30
LAM = 1.0


def comparar_escenario(nombre, crear_escenario, lam=LAM, repeticiones=REPETICIONES):
    """Ejecuta todos los algoritmos en un escenario y mide el tiempo mediano."""
    t, i, r = crear_escenario()
    resultados = [alg(t, i, r, lam=lam) for alg in ALGORITMOS]
    tabla = tabla_resultados(resultados, t, i, r, lam).drop(columns="Tiempo (ms)")
    tabla["Tiempo mediano (ms)"] = [
        round(statistics.median(alg(t, i, r, lam=lam).tiempo_ms for _ in range(repeticiones)), 3)
        for alg in ALGORITMOS
    ]
    tabla.insert(0, "Escenario", nombre)
    return tabla


comparacion = pd.concat([comparar_escenario(n, f) for n, f in ESCENARIOS.items()])
comparacion

In [ ]:
COLOR_BARRA = "#2a78d6"

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained", sharey=True)
for ax, (nombre, datos) in zip(ejes, comparacion.groupby("Escenario", sort=False)):
    barras = ax.barh(list(datos.index), datos["Nodos expandidos"], color=COLOR_BARRA, height=0.6)
    for barra, exp, costo in zip(barras, datos["Nodos expandidos"], datos["Costo"]):
        ax.text(barra.get_width() + 1.5, barra.get_y() + barra.get_height() / 2,
                f"{exp} nodos · costo {costo:g}", va="center", fontsize=8, color="#52514e")
    ax.set_title(nombre, loc="left", fontsize=10)
    ax.set_xlabel("Nodos expandidos")
    ax.set_xlim(0, datos["Nodos expandidos"].max() * 1.55)
    ax.grid(axis="x", color="#e5e4df", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(axis="y", length=0)

ejes[0].invert_yaxis()  # BFS arriba; con sharey basta invertir una vez
fig.suptitle("Trabajo realizado por cada algoritmo (λ = 1)", x=0.01, ha="left", fontsize=11)
plt.show()

**Conclusiones de la comparación**

| | Ruta conveniente (menor costo) | Poco trabajo |
|---|---|---|
| **BFS** | No: minimiza pasos, no riesgo | No |
| **DFS** | No: rutas muy largas | No |
| **UCS** | **Sí** | No |
| **Greedy** | No: va directo aunque cruce agua | **Sí** |
| **A\*** | **Sí** | **Sí**, comparado con UCS |

1. **Solo UCS y A\* encuentran la ruta conveniente**, porque son los únicos que incluyen el riesgo en el costo que optimizan. En ambos escenarios obtienen exactamente el mismo costo (19 y 15).
2. **A\* hace aproximadamente la mitad del trabajo que UCS:** 48 contra 89 nodos expandidos en el escenario principal, y 38 contra 79 en el de dos refugios. Es la mejor estrategia del prototipo.
3. **Greedy es el más rápido, pero no es confiable:** expande el mínimo de nodos y elige la misma ruta riesgosa que BFS.
4. En **dos refugios**, BFS y Greedy eligen el refugio cercano cruzando el agua (costo 16). UCS y A\* eligen el refugio lejano por terreno seco (costo 15). El riesgo cambió **a qué refugio** conviene ir, no solo por dónde.
5. **El tiempo depende del equipo y no siempre sigue a los nodos expandidos.** Por eso los nodos expandidos son la medida de trabajo más justa entre algoritmos.

### 5.1 Efecto del peso del riesgo (λ)
_Responsable: Gildardo Emmanuel González García_

Se ejecutan UCS y A\* en el escenario principal con **61 valores de λ** (de 0 a 3, cada 0.05) y se registra qué ruta elige cada uno.

El objetivo es ver con ejecuciones reales:
1. en qué valores de λ cambia la decisión del algoritmo;
2. si A\* sigue obteniendo el mismo costo óptimo que UCS para cualquier λ.

In [ ]:
import math
from dataclasses import replace

terreno, inicio, refugios = escenario_atajo_inundado()
VALORES_LAMBDA = np.round(np.arange(0, 3.01, 0.05), 2)

filas = []
for lam in VALORES_LAMBDA:
    r_ucs = ucs(terreno, inicio, refugios, lam=float(lam))
    r_a = a_estrella(terreno, inicio, refugios, lam=float(lam))
    filas.append({
        "λ": float(lam),
        "Pasos": r_ucs.pasos,
        "Riesgo": r_ucs.riesgo_acumulado,
        "Costo UCS": r_ucs.costo,
        "Costo A*": r_a.costo,
        "Expandidos UCS": r_ucs.nodos_expandidos,
        "Expandidos A*": r_a.nodos_expandidos,
        "ruta": tuple(r_ucs.ruta),
    })
barrido = pd.DataFrame(filas)

mismo_costo = all(math.isclose(u, a) for u, a in zip(barrido["Costo UCS"], barrido["Costo A*"]))
print(f"Valores de λ probados: {len(barrido)}")
print(f"¿A* obtuvo el mismo costo que UCS en todos? {mismo_costo}")
print(f"Nodos expandidos promedio: UCS {barrido['Expandidos UCS'].mean():.1f}, "
      f"A* {barrido['Expandidos A*'].mean():.1f}")

# Filas donde la ruta elegida por UCS cambia respecto al λ anterior
cambios = barrido[barrido["ruta"] != barrido["ruta"].shift()]
cambios[["λ", "Pasos", "Riesgo", "Costo UCS", "Expandidos UCS", "Expandidos A*"]]

## 6. Pruebas
_Responsable: Anthony Oziel Ramos López_

Todas las rutas se comprueban con el validador (`src/validacion.py`).

### 6.1 Existe una ruta disponible hasta el refugio

### 6.2 Se bloquea una zona que formaba parte de la ruta

### 6.3 El refugio deja de ser accesible

### 6.4 Se modifica el riesgo de una región y se vuelve a buscar

### 6.5 El camino más corto deja de ser el más conveniente

### 6.6 Prueba adicional: detección de errores de implementación
En mapas aleatorios, A* y UCS deben obtener siempre el mismo costo óptimo. Si difieren, hay un error en la heurística o en la implementación.

## 7. Decisiones de diseño y conclusiones
_Responsable: Hector Partido Flores (con aportes de todos)_

- Por qué una cuadrícula y movimiento en 4 direcciones.
- Cómo se incorpora el riesgo en el costo.
- Respuesta a la pregunta guía, con base en los resultados de las secciones 5 y 6.
- Diferencias observadas entre estrategias.

## 8. Declaración de uso de herramientas de IA
_Responsable: Hector Partido Flores_

- Herramientas utilizadas y para qué.
- Cómo se verificó el código generado (pruebas, validador, revisión en Pull Requests).